<a href="https://colab.research.google.com/github/Deva2013/airline-disruption-management-system/blob/main/Airline_Disruption_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure ──────────────────────────────────────────────────
# Everything lives on LOCAL Colab disk (/content) — fast, reliable, no
# Drive quota issues. This does NOT persist across sessions; that's
# expected. Finished "gold" outputs get pushed to Hugging Face Hub at the
# end of each stage instead of being written to Drive.

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [2]:
# ── Pull the verified BTS dataset from Hugging Face Hub ────────────────────
from huggingface_hub import hf_hub_download

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

print(f"Downloaded to: {bts_path}")

# Quick verification
df = pd.read_parquet(bts_path)
print(f"\nRows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nColumn list: {df.columns.tolist()}")

bts_cleaned.parquet: reconstructing file:   0%|          |  0.00B /  355MB            

bts_cleaned.parquet: downloading bytes:           |  0.00B            

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet

Rows: 10,504,936
Columns: 41
Year range: 2022 - 2024

Column list: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Airline', 'FlightNumber', 'OperatingAirline', 'OperatingAirlineCode', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'TaxiOut', 'TaxiIn', 'CRSArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'AirTime', 'Distance', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'ScheduledDepHour', 'Season', 'IsWeekend']


In [3]:
# ── Dataset overview ────────────────────────────────────────────────────
total     = len(df)
cancelled = df['Cancelled'].sum()
dep_del   = df['DepDel15'].sum()
arr_del   = df['ArrDel15'].sum()

print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Flights       : {total:,}')
print(f'  Date range    : {df["FlightDate"].min()} → {df["FlightDate"].max()}')
print(f'  Airlines      : {df["Airline"].nunique()}')
print(f'  Cancellation  : {cancelled/total*100:.2f}%  ({cancelled:,})')
print(f'  Dep delay≥15m : {dep_del/total*100:.2f}%  ({dep_del:,})')
print(f'  Arr delay≥15m : {arr_del/total*100:.2f}%  ({arr_del:,})')
print()
print('Severity breakdown:')
print(df['SeverityTier'].value_counts().to_string())

  DATASET OVERVIEW
  Flights       : 10,504,936
  Date range    : 2022-01-01 00:00:00 → 2024-12-31 00:00:00
  Airlines      : 10
  Cancellation  : 1.81%  (190,145)
  Dep delay≥15m : 20.04%  (2,105,664)
  Arr delay≥15m : 20.45%  (2,148,534)

Severity breakdown:
SeverityTier
On Time        8166257
Minor          1140970
Significant     697812
Severe          309752
Cancelled       190145


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Fetch Historical METAR Weather Data (IEM ASOS Archive)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Pull hourly weather observations for all 10 hub airports,
#          2022-2024, from the Iowa Environmental Mesonet (IEM) ASOS
#          archive — a true historical data source (unlike
#          aviationweather.gov, which only serves the last 15 days).
#
# Storage: Raw per-airport-year CSVs are cached on LOCAL Colab disk
#          (BASE_DIR/data/metar_raw), NOT Google Drive. This data is
#          disposable/re-fetchable, so it doesn't need to persist
#          beyond this session.
# ═══════════════════════════════════════════════════════════════════════

import io

IEM_URL = 'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py'
METAR_RAW = BASE_DIR / 'data' / 'metar_raw'
METAR_RAW.mkdir(exist_ok=True)

# Same 10 hub airports used in the BTS dataset, mapped to ICAO codes
# (ICAO codes are the 4-letter identifiers IEM's station data references)
AIRPORT_ICAO = {
    'JFK': 'KJFK', 'ORD': 'KORD', 'ATL': 'KATL', 'LAX': 'KLAX', 'DFW': 'KDFW',
    'SFO': 'KSFO', 'EWR': 'KEWR', 'MIA': 'KMIA', 'SEA': 'KSEA', 'BOS': 'KBOS',
}
START_YEAR, END_YEAR = 2022, 2024


def iem_station(icao):
    """IEM station IDs drop the leading 'K' for CONUS airports (KJFK -> JFK)."""
    return icao[1:] if icao.startswith('K') and len(icao) == 4 else icao


def fetch_iem_year(icao, year, max_retries=3):
    """
    Fetch one full year of METAR data for one airport from IEM.
    Retries with backoff on failure, and validates the response is
    real CSV data (not an HTML error/rate-limit page) before returning it.
    """
    params = [
        ('station', iem_station(icao)),
        ('data', 'all'),
        ('tz', 'Etc/UTC'),
        ('format', 'comma'),
        ('latlon', 'no'),
        ('sts', f'{year}-01-01T00:00:00Z'),
        ('ets', f'{year+1}-01-01T00:00:00Z' if year < END_YEAR else f'{year}-12-31T23:59:59Z'),
    ]
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(
                IEM_URL, params=params,
                headers={'User-Agent': 'airline-disruption-research (ASU student project)'},
                timeout=90
            )
            r.raise_for_status()
            text = r.text

            # Validation: IEM prefixes real responses with a few "#DEBUG:"
            # comment lines before the actual CSV header. Check the header
            # shows up near the top rather than requiring it be line 1 —
            # this rejects bad/empty responses instead of crashing later.
            if 'station,valid' in text[:600]:
                return text
            else:
                print(f'    [warn] {icao} {year}: unexpected response '
                      f'(attempt {attempt}), first 80 chars: {text[:80]!r}')
        except Exception as e:
            print(f'    [warn] {icao} {year}: {e} (attempt {attempt})')
        time.sleep(3 * attempt)  # back off longer with each retry
    return None


# ── Main fetch loop: one request per airport per year (30 total) ──────────
print('Fetching METAR data from IEM ASOS archive (local disk cache)...')
all_frames = []
failed = []

for iata, icao in AIRPORT_ICAO.items():
    print(f'  {iata} ({icao})')
    for year in range(START_YEAR, END_YEAR + 1):
        cache_file = METAR_RAW / f'{icao}_{year}.csv'

        # Reuse cached file if it exists and looks valid; otherwise fetch fresh
        if cache_file.exists():
            text = cache_file.read_text()
            if 'station,valid' not in text[:600]:
                cache_file.unlink()  # discard invalid cached file
                text = None
        else:
            text = None

        if text is None:
            text = fetch_iem_year(icao, year)
            if text:
                cache_file.write_text(text)
            time.sleep(1.5)  # be polite to IEM's server between requests

        if not text:
            failed.append((icao, year))
            continue

        # comment='#' skips IEM's "#DEBUG:" preamble lines automatically
        df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')
        df_year['IATA'] = iata
        all_frames.append(df_year)

# ── Combine all airport-years into one dataframe ───────────────────────────
wx_raw = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
print(f'\nTotal METAR records: {len(wx_raw):,}')
if failed:
    print(f'⚠️  Failed after retries: {failed}')
wx_raw.head(2)

Fetching METAR data from IEM ASOS archive (local disk cache)...
  JFK (KJFK)
  ORD (KORD)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 1)
    [warn] KORD 2023: 503 Server Error: Service Unavailable for url: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=ORD&data=all&tz=Etc%2FUTC&format=comma&latlon=no&sts=2023-01-01T00%3A00%3A00Z&ets=2024-01-01T00%3A00%3A00Z (attempt 2)
  ATL (KATL)
  LAX (KLAX)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  DFW (KDFW)
  SFO (KSFO)
  EWR (KEWR)
  MIA (KMIA)


/tmp/ipykernel_1177/1516899102.py:105: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(io.StringIO(text), na_values=['M'], comment='#')


  SEA (KSEA)
  BOS (KBOS)

Total METAR records: 3,381,319


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth,IATA
0,JFK,2022-01-01 00:00,49.0,48.0,96.32,200.0,6.0,NaN,NaN,1014.9,...,NaN,NaN,NaN,NaN,NaN,NaN,46.0,METAR JFK 010000Z AUTO 20006KT BR 09/09 RMK AO...,NaN,JFK
1,JFK,2022-01-01 00:00,NaN,NaN,NaN,200.0,5.0,NaN,29.97,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KJFK 010000Z AUTO 20005KT 8SM BKN004 OVC014 09...,NaN,JFK


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Clean and Aggregate METAR Data to Hourly Airport-Level Records
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Convert raw ~5-minute-interval METAR observations into one
#          clean row per (airport, date, hour) — matching the grain
#          needed to join against the BTS flight data later.
#
# Key logic: IEM reports every ~5 minutes, so each airport-hour has
#            multiple raw observations. We aggregate to hourly by
#            keeping the WORST (most disruptive) condition seen in
#            that hour — e.g. minimum visibility, maximum wind —
#            since that's what actually matters for flight delays.
# ═══════════════════════════════════════════════════════════════════════

def clean_metars(df):
    df = df.copy()

    # Parse observation timestamp; drop rows where it's unparseable
    df['ObsTime'] = pd.to_datetime(df['valid'], errors='coerce', utc=True)
    df = df.dropna(subset=['ObsTime'])

    # Convert/rename raw IEM fields to clean, analysis-ready columns
    df['TempC']       = (pd.to_numeric(df['tmpf'], errors='coerce') - 32) * 5 / 9
    df['WindSpeedKt'] = pd.to_numeric(df['sknt'], errors='coerce')
    df['WindGustKt']  = pd.to_numeric(df['gust'], errors='coerce')
    df['VisibSM']     = pd.to_numeric(df['vsby'], errors='coerce')

    # Disruption indicator flags (thresholds based on common aviation
    # operational impact levels)
    df['IsLowVis']   = (df['VisibSM']    <  3).astype(int)
    df['IsHighWind'] = (df['WindSpeedKt']>= 25).astype(int)
    df['IsGust']     = (df['WindGustKt'] >= 35).astype(int)

    # Extract date and hour for joining against flight-level BTS data
    df['Date']    = df['ObsTime'].dt.date
    df['DepHour'] = df['ObsTime'].dt.hour

    # ── Aggregate multiple observations per airport-hour into one row ──
    # Uses the WORST condition seen in the hour (min visibility, max
    # wind/gust) since that's most relevant to whether flights were
    # actually disrupted during that hour.
    agg = df.groupby(['IATA', 'Date', 'DepHour']).agg(
        TempC=('TempC', 'mean'),
        WindSpeedKt=('WindSpeedKt', 'max'),
        WindGustKt=('WindGustKt', 'max'),
        VisibSM=('VisibSM', 'min'),
        IsLowVis=('IsLowVis', 'max'),
        IsHighWind=('IsHighWind', 'max'),
        IsGust=('IsGust', 'max'),
    ).reset_index()

    # Classify flight-rules category based on worst visibility in the hour
    def flight_cat(vis):
        if pd.isna(vis): return 'Unknown'
        if vis < 1:  return 'LIFR'   # Low Instrument Flight Rules
        if vis < 3:  return 'IFR'    # Instrument Flight Rules
        if vis < 5:  return 'MVFR'   # Marginal Visual Flight Rules
        return 'VFR'                 # Visual Flight Rules (normal)
    agg['FlightCategory'] = agg['VisibSM'].apply(flight_cat)
    agg['IsFogOrIFR'] = agg['FlightCategory'].isin(['IFR', 'LIFR']).astype(int)

    return agg


# ── Run cleaning and save locally ──────────────────────────────────────
wx = clean_metars(wx_raw)

wx_path = DIR_WEATHER / 'metar_clean.parquet'
wx.to_parquet(wx_path, index=False)

print(f'Clean METAR records (one row per airport-hour): {len(wx):,}')
print(f'Saved locally to: {wx_path}')
print()
print('Flight category breakdown:')
print(wx['FlightCategory'].value_counts().to_string())

Clean METAR records (one row per airport-hour): 263,034
Saved locally to: /content/airline-disruption/data/weather/metar_clean.parquet

Flight category breakdown:
FlightCategory
VFR     243922
MVFR      7746
IFR       7493
LIFR      3873


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload Clean METAR Dataset to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Persist the finished, cleaned weather dataset permanently.
#          This is a "gold" file (small, finished, analysis-ready) —
#          unlike the raw METAR CSVs, which stay local/disposable.
# ═══════════════════════════════════════════════════════════════════════

hf_api.upload_file(
    path_or_fileobj=str(wx_path),
    path_in_repo="metar_clean.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print("✅ metar_clean.parquet uploaded to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ather/metar_clean.parquet:  72%|#######1  |  536kB /  745kB            

✅ metar_clean.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Join BTS Flight Data with METAR Weather Data
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Attach weather conditions to each flight, based on the
#          ORIGIN airport and the flight's SCHEDULED departure hour.
#          This is what lets the disruption engine later distinguish
#          weather-driven delays from other causes.
#
# Join keys: Origin airport (IATA) + FlightDate + ScheduledDepHour
#            match against wx's IATA + Date + DepHour.
# ═══════════════════════════════════════════════════════════════════════

# ── Prepare join keys on both sides ────────────────────────────────────
# BTS side: use the flight's origin airport, date, and scheduled dep hour
df_join = df.copy()
df_join['JoinDate'] = df_join['FlightDate'].dt.date  # match wx's date type

# Weather side: already has IATA, Date, DepHour from the cleaning step
wx_join = wx.copy()

# ── Perform the join ────────────────────────────────────────────────────
# Left join: keep ALL flights, even if weather data is missing for that
# airport-hour (rare given ~100% coverage, but don't want to silently
# drop flights if it happens)
bts_weather = df_join.merge(
    wx_join,
    left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'],
    how='left'
)

# Drop the now-redundant join key columns from the weather side
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])

# ── Check join quality ──────────────────────────────────────────────────
matched = bts_weather['FlightCategory'].notna().sum()
total = len(bts_weather)

print(f'Total flights: {total:,}')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched (no weather record for that airport-hour): {total - matched:,}')
print()
print('Sample of joined data:')
bts_weather[['FlightDate', 'Origin', 'ScheduledDepHour', 'TempC',
             'WindSpeedKt', 'VisibSM', 'FlightCategory']].head()

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Set up local working directories and authenticate with
#          Hugging Face Hub. No Google Drive is used anywhere in this
#          pipeline — all working data lives on local Colab disk
#          (/content), and finished outputs are pushed to Hugging Face
#          Hub for permanent storage.
# ═══════════════════════════════════════════════════════════════════════

import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

# Install Hugging Face Hub client
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login, hf_hub_download

# ── Authenticate with Hugging Face using the Colab secret ──────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

# ── Directory structure (local Colab disk only) ─────────────────────────
BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Base directory: {BASE_DIR}')
print(f'HF repo target: {HF_REPO_ID}')

Environment ready.
Base directory: /content/airline-disruption
HF repo target: Dev123Hug456Face/airline-disruption-data


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull BTS Flight Data from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the cleaned BTS dataset (previously uploaded) into
#          this fresh session, without needing to reprocess the 36
#          raw ZIP files again.
# ═══════════════════════════════════════════════════════════════════════

bts_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_cleaned.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

df = pd.read_parquet(bts_path)

print(f"Downloaded to: {bts_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Downloaded to: /content/airline-disruption/data/processed/bts_cleaned.parquet
Rows: 10,504,936
Columns: 41


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Clean METAR Weather Data from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the cleaned, hourly-aggregated weather dataset
#          (previously uploaded) into this fresh session, without
#          needing to re-fetch from IEM's ASOS archive again.
# ═══════════════════════════════════════════════════════════════════════

wx_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="metar_clean.parquet",
    repo_type="dataset",
    local_dir=DIR_WEATHER,
)

wx = pd.read_parquet(wx_path)

print(f"Downloaded to: {wx_path}")
print(f"Rows: {len(wx):,}")
print(f"Columns: {wx.shape[1]}")

Downloaded to: /content/airline-disruption/data/weather/metar_clean.parquet
Rows: 263,034
Columns: 12


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Reduce Memory Footprint Before Joining (prevents OOM crash)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The previous join attempt crashed from running out of RAM.
#          Root cause: several BTS columns are stored as generic 'object'
#          (string) dtype, which is memory-heavy in pandas — especially
#          across 10.5M rows. Converting repetitive text columns to
#          'category' dtype stores each unique value once instead of
#          once per row, cutting memory use dramatically before we
#          attempt the merge.
# ═══════════════════════════════════════════════════════════════════════

import gc

# Columns with a small number of repeated values — ideal for 'category'
category_cols = [
    'Airline', 'OperatingAirline', 'OperatingAirlineCode',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'CancellationReason', 'PrimaryDelayCause', 'SeverityTier', 'Season',
]

for col in category_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Same treatment for the weather side
if 'FlightCategory' in wx.columns:
    wx['IATA'] = wx['IATA'].astype('category')
    wx['FlightCategory'] = wx['FlightCategory'].astype('category')

# Force garbage collection to release any freed memory immediately
gc.collect()

print('Memory optimization complete.')
print(f'df memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'wx memory usage: {wx.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Memory optimization complete.
df memory usage: 1.51 GB
wx memory usage: 0.03 GB


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Check Available Memory, Then Join BTS + Weather Data
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Confirm there's enough RAM headroom before attempting the
#          merge (which crashed the session last time before memory
#          optimization). Then join BTS flights with weather data,
#          matching each flight's ORIGIN airport + FlightDate +
#          ScheduledDepHour against the weather record for that
#          airport-hour.
# ═══════════════════════════════════════════════════════════════════════

# Check available RAM first
!free -h

print()
print('Proceeding with join...')

# ── Prepare join keys ────────────────────────────────────────────────────
df['JoinDate'] = df['FlightDate'].dt.date  # match wx's date type

# ── Perform the join (left join: keep ALL flights) ─────────────────────
bts_weather = df.merge(
    wx,
    left_on=['Origin', 'JoinDate', 'ScheduledDepHour'],
    right_on=['IATA', 'Date', 'DepHour'],
    how='left'
)

# Drop redundant join key columns from the weather side
bts_weather = bts_weather.drop(columns=['JoinDate', 'IATA', 'Date', 'DepHour'])

# Free up memory from intermediate objects no longer needed
gc.collect()

# ── Check join quality ──────────────────────────────────────────────────
matched = bts_weather['FlightCategory'].notna().sum()
total = len(bts_weather)

print(f'\nTotal flights: {total:,}')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched: {total - matched:,}')
print(f'\nResult memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')

               total        used        free      shared  buff/cache   available
Mem:            12Gi       8.1Gi       2.7Gi       2.0Mi       1.8Gi       4.3Gi
Swap:             0B          0B          0B

Proceeding with join...

Total flights: 10,504,936
Matched with weather data: 5,896,424 (56.13%)
Unmatched: 4,608,512

Result memory usage: 2.72 GB


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Filter to Origin-Hub Flights, Then Re-Verify Weather Join
# ═══════════════════════════════════════════════════════════════════════
# Purpose: The previous join only matched 56% of flights, because the
#          BTS dataset includes flights arriving AT a hub from
#          non-hub origins — and weather data only covers the 10 hub
#          airports. Restricting to flights DEPARTING FROM a hub
#          (matching the project's departure-disruption framing)
#          should give near-100% weather match coverage.
# ═══════════════════════════════════════════════════════════════════════

TARGET_AIRPORTS = ['JFK', 'ORD', 'ATL', 'LAX', 'DFW', 'SFO', 'EWR', 'MIA', 'SEA', 'BOS']

# Keep only flights departing FROM one of the 10 target hub airports
bts_weather_scoped = bts_weather[bts_weather['Origin'].isin(TARGET_AIRPORTS)].copy()

matched = bts_weather_scoped['FlightCategory'].notna().sum()
total = len(bts_weather_scoped)

print(f'Flights departing from target hubs: {total:,} (was {len(bts_weather):,} before filtering)')
print(f'Matched with weather data: {matched:,} ({matched/total*100:.2f}%)')
print(f'Unmatched: {total - matched:,}')

Flights departing from target hubs: 5,896,606 (was 10,504,936 before filtering)
Matched with weather data: 5,896,424 (100.00%)
Unmatched: 182


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Free Memory and Save the Joined Dataset Locally
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Drop the old unfiltered join result (no longer needed) to
#          free memory, then save the final scoped BTS+weather dataset
#          to local disk as the "gold" output for this stage.
# ═══════════════════════════════════════════════════════════════════════

# Free the old unfiltered join — no longer needed, was using ~2.7GB
del bts_weather
gc.collect()

# Rename for clarity going forward
bts_weather = bts_weather_scoped
del bts_weather_scoped
gc.collect()

# Save locally first
joined_path = DIR_PROCESSED / 'bts_weather_joined.parquet'
bts_weather.to_parquet(joined_path, index=False)

print(f'Saved: {joined_path}')
print(f'Rows: {len(bts_weather):,}')
print(f'Columns: {bts_weather.shape[1]}')
print(f'Memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Saved: /content/airline-disruption/data/processed/bts_weather_joined.parquet
Rows: 5,896,606
Columns: 50
Memory usage: 1.57 GB


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Upload Joined BTS+Weather Dataset to Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Persist the finished BTS+weather join permanently — this is
#          the "gold" dataset the disruption detection engine and
#          predictive model will build on next.
# ═══════════════════════════════════════════════════════════════════════

hf_api.upload_file(
    path_or_fileobj=str(joined_path),
    path_in_repo="bts_weather_joined.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)

print("✅ bts_weather_joined.parquet uploaded to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ts_weather_joined.parquet:   1%|          |  560kB /  112MB            

✅ bts_weather_joined.parquet uploaded to Hugging Face Hub


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Classify each flight's disruption using both the BTS-reported
#          delay cause (CarrierDelay, WeatherDelay, NASDelay, etc.) and
#          the actual weather conditions at departure (from our METAR
#          join) to catch cases where weather contributed to a delay
#          even when BTS attributed it to a different primary cause
#          (a known reporting quirk — airlines often under-report
#          weather as the cause when it's a contributing factor).
# ═══════════════════════════════════════════════════════════════════════

# ── Weather-Contributed Flag ────────────────────────────────────────────
# True if degraded weather conditions were present at departure time,
# REGARDLESS of what BTS listed as the "official" primary cause.
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

# ── Refined Disruption Classification ───────────────────────────────────
# Combines BTS's official PrimaryDelayCause with our own weather signal
# to produce a more complete disruption type label.
def classify_disruption(row):
    if row['Cancelled'] == 1:
        return 'Cancelled'
    if row['Diverted'] == 1:
        return 'Diverted'
    if row['ArrDelayMinutes'] < 15:
        return 'On Time'
    # Delayed flight — determine primary driver
    if row['WeatherContributed'] == 1:
        return 'Weather-Related Delay'
    if pd.notna(row['PrimaryDelayCause']):
        return f"{row['PrimaryDelayCause']} Delay"
    return 'Other Delay'

bts_weather['DisruptionType'] = bts_weather.apply(classify_disruption, axis=1)

# ── Cascade Risk Flag ────────────────────────────────────────────────────
# Flights with LateAircraftDelay > 0 indicate the delay is a downstream
# effect of a PREVIOUS flight's lateness (the incoming aircraft was late)
# — this is a useful signal for identifying cascading disruption chains.
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

# ── Summary ──────────────────────────────────────────────────────────────
print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())
print()
print(f'Weather-contributed flights: {bts_weather["WeatherContributed"].sum():,} '
      f'({bts_weather["WeatherContributed"].mean()*100:.2f}%)')
print(f'Cascade-risk flights (late aircraft): {bts_weather["IsCascadeRisk"].sum():,} '
      f'({bts_weather["IsCascadeRisk"].mean()*100:.2f}%)')

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Environment Setup
# ═══════════════════════════════════════════════════════════════════════
import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

!pip install -q huggingface_hub

from huggingface_hub import HfApi, login, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_REPO_ID = "Dev123Hug456Face/airline-disruption-data"
hf_api = HfApi()

BASE_DIR       = Path('/content/airline-disruption')
DIR_RAW        = BASE_DIR / 'data' / 'raw'
DIR_PROCESSED  = BASE_DIR / 'data' / 'processed'
DIR_WEATHER    = BASE_DIR / 'data' / 'weather'

for d in [DIR_RAW, DIR_PROCESSED, DIR_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

print('Environment ready.')

Environment ready.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Pull Joined BTS+Weather Dataset from Hugging Face Hub
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Restore the already-joined, origin-hub-scoped dataset
#          (5.9M flights) directly — skips re-downloading BTS,
#          re-fetching weather, and re-running the join.
# ═══════════════════════════════════════════════════════════════════════

joined_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="bts_weather_joined.parquet",
    repo_type="dataset",
    local_dir=DIR_PROCESSED,
)

bts_weather = pd.read_parquet(joined_path)

print(f"Downloaded to: {joined_path}")
print(f"Rows: {len(bts_weather):,}")
print(f"Columns: {bts_weather.shape[1]}")
print(f"Memory usage: {bts_weather.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Downloaded to: /content/airline-disruption/data/processed/bts_weather_joined.parquet
Rows: 5,896,606
Columns: 50
Memory usage: 1.52 GB


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Disruption Detection Engine (Vectorized — Memory-Safe)
# ═══════════════════════════════════════════════════════════════════════
# Purpose: Classify each flight's disruption type using both the BTS-
#          reported delay cause and actual weather conditions at
#          departure. Rewritten using vectorized numpy operations
#          instead of df.apply(axis=1) — the row-by-row .apply() call
#          is what crashed the session last time; np.select() performs
#          the same logic across all 5.9M rows at once, using far less
#          memory and running dramatically faster.
# ═══════════════════════════════════════════════════════════════════════

# ── Weather-Contributed Flag ────────────────────────────────────────────
# True if degraded weather conditions were present at departure time,
# regardless of what BTS listed as the "official" primary cause.
bts_weather['WeatherContributed'] = (
    (bts_weather['IsLowVis'] == 1) |
    (bts_weather['IsHighWind'] == 1) |
    (bts_weather['IsGust'] == 1) |
    (bts_weather['IsFogOrIFR'] == 1)
).astype(int)

# ── Refined Disruption Classification (vectorized) ──────────────────────
# Conditions checked in priority order — first matching condition wins,
# same logic as the row-by-row version, just evaluated column-wise
# across the whole dataset at once instead of row-by-row.
conditions = [
    bts_weather['Cancelled'] == 1,
    bts_weather['Diverted'] == 1,
    bts_weather['ArrDelayMinutes'] < 15,
    bts_weather['WeatherContributed'] == 1,
    bts_weather['PrimaryDelayCause'].notna(),
]

choices = [
    'Cancelled',
    'Diverted',
    'On Time',
    'Weather-Related Delay',
    bts_weather['PrimaryDelayCause'].astype(str) + ' Delay',
]

bts_weather['DisruptionType'] = np.select(conditions, choices, default='Other Delay')

# ── Cascade Risk Flag ────────────────────────────────────────────────────
# Flights with LateAircraftDelay > 0 indicate the delay is a downstream
# effect of a previous flight's lateness (incoming aircraft was late).
bts_weather['IsCascadeRisk'] = (bts_weather['LateAircraftDelay'] > 0).astype(int)

# ── Summary ──────────────────────────────────────────────────────────────
print('Disruption Type breakdown:')
print(bts_weather['DisruptionType'].value_counts().to_string())
print()
print(f'Weather-contributed flights: {bts_weather["WeatherContributed"].sum():,} '
      f'({bts_weather["WeatherContributed"].mean()*100:.2f}%)')
print(f'Cascade-risk flights (late aircraft): {bts_weather["IsCascadeRisk"].sum():,} '
      f'({bts_weather["IsCascadeRisk"].mean()*100:.2f}%)')

Disruption Type breakdown:
DisruptionType
On Time                  4525651
Carrier Delay             432795
Late Aircraft Delay       392168
NAS Delay                 288989
Cancelled                 104301
Weather-Related Delay      99051
Weather Delay              36421
Diverted                   14775
Security Delay              2453
Other Delay                    2

Weather-contributed flights: 319,563 (5.42%)
Cascade-risk flights (late aircraft): 564,889 (9.58%)


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Save Disruption-Classified Dataset Locally, Then Upload to Hub
# ═══════════════════════════════════════════════════════════════════════
detected_path = DIR_PROCESSED / 'bts_disruption_detected.parquet'
bts_weather.to_parquet(detected_path, index=False)
print(f'Saved: {detected_path}')

hf_api.upload_file(
    path_or_fileobj=str(detected_path),
    path_in_repo="bts_disruption_detected.parquet",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
print("✅ bts_disruption_detected.parquet uploaded to Hugging Face Hub")

Saved: /content/airline-disruption/data/processed/bts_disruption_detected.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sruption_detected.parquet:  14%|#3        | 16.0MB /  115MB            

✅ bts_disruption_detected.parquet uploaded to Hugging Face Hub
